# LLM Zoomcamp HW4 · Evaluation

行动清单：[hw4-today.md](../learning-plan-2026/hw4-today.md)

在 HW2 同一项目继续。需要 `.env` 里的 `OPENAI_API_KEY`（仅 Q1）。

## Setup · 加载数据 + 索引

In [1]:
from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from gitsource import GithubRepositoryDataReader, chunk_documents
from embedder import Embedder
from minsearch import Index, VectorSearch

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [f.parse() for f in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [2]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

embed = Embedder()
texts = [c["content"] for c in chunks]
vectors = []
for i in tqdm(range(0, len(texts), 50)):
    vectors.extend(embed.encode_batch(texts[i:i + 50]))
X = np.array(vectors)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

  0%|          | 0/6 [00:00<?, ?it/s]

In [3]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    return vindex.search(embed.encode(query), num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    return rrf([text_search(query, 10), vector_search(query, 10)], k=k)

## Q1 · 为前 3 页生成问题

In [4]:
import json
from pydantic import BaseModel
from openai import OpenAI
from evaluation_utils import llm_structured

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

client = OpenAI()
target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
pages = [d for d in documents if d["filename"] in target_files]

input_tokens_list = []
for page in pages:
    _, usage = llm_structured(client, data_gen_instructions, json.dumps(page), Questions)
    input_tokens_list.append(usage.input_tokens)
    print(page["filename"], usage.input_tokens)

sum(input_tokens_list) / len(input_tokens_list)

01-agentic-rag/lessons/01-intro.md 1020
01-agentic-rag/lessons/02-environment.md 1286
01-agentic-rag/lessons/03-rag.md 1753


1353.0

## Ground Truth + Q2/Q3

In [5]:
ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
len(ground_truth)

q = ground_truth[0]["question"]
ground_truth[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [6]:
text_search(q)[0]["filename"]  

'01-agentic-rag/lessons/03-rag.md'

In [7]:
vector_search(q)[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

## Q4–Q6 · evaluate

In [8]:
def compute_relevance(q, search_fn):
    target = q["filename"]
    return [int(r["filename"] == target) for r in search_fn(q["question"])]

def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)

def mrr(relevance):
    total = 0.0
    for line in relevance:
        for rank, v in enumerate(line):
            if v == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance)

def evaluate(search_fn):
    rel = [compute_relevance(q, search_fn) for q in ground_truth]
    return {"hit_rate": hit_rate(rel), "mrr": mrr(rel)}

In [9]:
evaluate(text_search)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [10]:
evaluate(vector_search)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [11]:
for k in [1, 50, 100, 200]:
    m = evaluate(lambda q, k=k: hybrid_search(q, k))["mrr"]
    print(f"k={k}: mrr={m:.3f}")  # Q6: 最高 MRR，并列取最小 k

k=1: mrr=0.648
k=50: mrr=0.638
k=100: mrr=0.638
k=200: mrr=0.638
